# SimPO training on ChartQA (method 5/7)

Trains on the exact same ChartQA DPO pairs file as the Full DPO run
(`experiments/020_chartqa_transfer/data/dpo_pairs.jsonl`, 127 pairs) --
training method is the only variable, matching the original project's SimPO
methodology exactly (same rationale: SimPO is reference-free and length-
normalized, beta=2.0 vs DPO's 0.1, per princeton-nlp/SimPO's tuning guidance).


In [ ]:
import subprocess, sys

gpu_names = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip().splitlines()
print('Detected GPUs (nvidia-smi):', gpu_names)
is_p100 = any('P100' in n for n in gpu_names)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

if is_p100:
    print('*** Tesla P100 detected. Installing the validated older stack...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow'], check=True)
else:
    print('Non-P100 GPU: upgrading transformers to a current release, leaving torch/peft/accelerate at image defaults.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.49.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft==0.14.0', 'qwen-vl-utils==0.0.14'], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, transformers: {transformers.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
import os
from pathlib import Path
import subprocess

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

pairs_path = repo_dir / 'experiments/020_chartqa_transfer/data/dpo_pairs.jsonl'
assert pairs_path.exists(), f'Missing {pairs_path} -- commit/push problem.'
with open(pairs_path, encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
print(f'Confirmed {n} ChartQA DPO pairs present (same file Full DPO trained on).')


In [ ]:
import sys

env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_simpo.py',
    '--dataset-path', 'experiments/020_chartqa_transfer/data/dpo_pairs.jsonl',
    '--images-dir', 'data/ChartQA/images',
    '--output-dir', '/kaggle/working/qwen_vl_simpo_chartqa_adapter',
    '--epochs', '1',
    '--batch-size', '1',
    '--lr', '1e-6',
    '--beta', '2.0',
    '--gamma-beta-ratio', '0.5',
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
out_dir = Path('/kaggle/working/qwen_vl_simpo_chartqa_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'SimPO ChartQA adapter directory {out_dir} contents: {files}')
assert (out_dir / 'adapter_config.json').exists(), 'adapter_config.json missing -- training did not save a LoRA adapter.'
